# Trabalho – Dados Faltantes sem Imputação Manual

Dataset: employees_dataset_with_missing.csv.xls

Objetivo: utilizar estimadores do scikit-learn que aceitam valores NaN diretamente, conforme solicitado.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('employees_dataset_with_missing.csv.xls')
print(df.shape)
df.head()


In [ ]:
# Quantidade de valores faltantes
missing = df.isna().sum().to_frame('faltantes')
missing['percentual'] = 100 * missing['faltantes'] / len(df)
missing


## Separação dos dados

Variável alvo: `credit_score`

As demais colunas serão usadas como preditoras.


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['credit_score'])
y = df['credit_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


## Modelos que aceitam NaN diretamente

Serão avaliados:

- HistGradientBoostingRegressor
- RandomForestRegressor
- ExtraTreesRegressor
- DecisionTreeRegressor


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor

modelos = {
    'HistGradientBoostingRegressor': HistGradientBoostingRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(random_state=42, n_estimators=200),
    'ExtraTreesRegressor': ExtraTreesRegressor(random_state=42, n_estimators=200),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42)
}

resultados = []

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    resultados.append({
        'Modelo': nome,
        'MAE': mean_absolute_error(y_test, pred),
        'R2': r2_score(y_test, pred)
    })

pd.DataFrame(resultados).sort_values('R2', ascending=False)


## Transformadores compatíveis com valores ausentes

Exemplos solicitados pelo professor:
- MinMaxScaler
- RobustScaler
- StandardScaler
- MissingIndicator


In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.impute import MissingIndicator

scalers = {
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
    'StandardScaler': StandardScaler()
}

for nome, scaler in scalers.items():
    Xt = scaler.fit_transform(X)
    print(nome, Xt.shape)

indicator = MissingIndicator()
indicadores = indicator.fit_transform(X)

print('MissingIndicator:', indicadores.shape)


## Conclusão

O experimento utiliza algoritmos capazes de trabalhar diretamente com valores ausentes (NaN), evitando estratégias tradicionais de imputação.

Os resultados permitem comparar diferentes estimadores compatíveis com dados faltantes e identificar o modelo com melhor desempenho para prever `credit_score`.
